# 02 - Lọc trùng lặp & Phân chia Train / Val / Test (Khoá tập Test)

**Bài tập lớn môn Học máy:** Phân loại 4 nhóm ảnh X-quang ngực (COVID-19 Radiography Database).  
**Tuần 3 (Sprint 2):** Lọc ảnh trùng hệt và gần trùng bằng imagehash (pHash), gom nhóm `group_id`, chia tập stratified 70/15/15 và khoá tập test.  
**Người phụ trách chính (Code Owner):** TV1 (với sự hỗ trợ kèm cặp của TV2).  

## Mục tiêu Tuần 3 (Track A):
1. Đọc file `manifest.csv` (21.165 ảnh).
2. Tính toán mã hash file (MD5) và perceptual hash (pHash) cho toàn bộ 21.165 ảnh.
3. Thử nghiệm và so sánh 3 ngưỡng khoảng cách Hamming ứng viên (0, 4, 8) để phát hiện ảnh gần trùng:
   - Báo cáo số nhóm trùng lặp, số ảnh bị gộp và trực quan hóa 6 cặp ảnh mẫu cho mỗi ngưỡng.
4. Gom nhóm ảnh trùng lặp (`group_id`) và phân chia train/val/test theo tỷ lệ 70/15/15 phân tầng theo lớp VÀ theo nhóm trùng lặp (Stratified Group Splitting).
5. Đảm bảo nguyên tắc chống rò rỉ (Leakage Prevention): Không có ảnh nào cùng nhóm trùng lặp nằm ở 2 tập khác nhau.
6. Xuất các candidate splits (`split_th0.csv`, `split_th4.csv`, `split_th8.csv`), `split_v1.csv`, tính mã băm `split_v1.sha256` và ghi nhận giới hạn về metadata bệnh nhân.


In [ ]:
import os
import sys
import hashlib
import shutil
import json
from pathlib import Path
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import imagehash
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.auto import tqdm

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("ImageHash:", imagehash.__version__)


In [ ]:
# 1. Định vị thư mục dataset và file manifest.csv
input_root = Path("/kaggle/input")
if not input_root.exists():
    input_root = Path(".")

print(f"Quét thư mục gốc: {input_root.resolve()}")

# Tìm thư mục chứa ảnh (target_root)
target_root = None
candidates = [input_root] + [d for d in input_root.rglob("*") if d.is_dir()]
for d in candidates:
    if (d / "COVID").is_dir() and ((d / "COVID" / "images").is_dir() or (d / "Normal").is_dir()):
        target_root = d
        break

if target_root is None:
    for d in candidates:
        if "radiography" in d.name.lower() and d.is_dir():
            target_root = d
            break

print(f"Thư mục ảnh (target_root): {target_root}")

# Tìm manifest.csv trong input hoặc thư mục cục bộ
manifest_path = None
for p in [Path("splits/manifest.csv"), Path("outputs/eda_manifest/manifest.csv"), Path("manifest.csv")]:
    if p.exists():
        manifest_path = p
        break

if manifest_path is None:
    for p in input_root.rglob("manifest.csv"):
        manifest_path = p
        break

if manifest_path is not None and manifest_path.exists():
    print(f"Nạp manifest thành công từ: {manifest_path.resolve()}")
    df_manifest = pd.read_csv(manifest_path)
else:
    print("CẢNH BÁO: Không tìm thấy manifest.csv có sẵn. Đang quét lại từ thư mục dataset...")
    records = []
    valid_exts = {'.png', '.jpg', '.jpeg'}
    class_dirs = sorted([d for d in target_root.iterdir() if d.is_dir() and any(k in d.name.lower() for k in ["covid", "normal", "lung", "opacity", "pneumonia"])])
    for cd in class_dirs:
        scan_dir = (cd / "images") if (cd / "images").is_dir() else cd
        masks_dir = cd / "masks"
        for f in sorted(scan_dir.iterdir()):
            if f.is_file() and f.suffix.lower() in valid_exts:
                m_p = masks_dir / f.name if masks_dir.is_dir() else None
                has_m = m_p.exists() if m_p else False
                records.append({
                    "image_id": f.name,
                    "label": cd.name,
                    "image_path": str(f.resolve()),
                    "rel_image_path": f"{cd.name}/images/{f.name}",
                    "mask_path": str(m_p.resolve()) if has_m else "",
                    "rel_mask_path": f"{cd.name}/masks/{f.name}" if has_m else "",
                    "has_mask": has_m,
                    "file_hash": hashlib.md5(open(f, "rb").read()).hexdigest()
                })
    df_manifest = pd.DataFrame(records)

print(f"Tổng số mẫu trong manifest: {len(df_manifest):,}")
print("Phân bố nhãn:")
print(df_manifest["label"].value_counts())


In [ ]:
# 2. Định nghĩa hàm tìm đường dẫn ảnh và tính pHash (64-bit DCT)
def resolve_img_path(row, root):
    p = Path(row["image_path"])
    if p.exists():
        return p
    if root is not None:
        p2 = root / row["rel_image_path"]
        if p2.exists():
            return p2
    p3 = Path(row["rel_image_path"])
    if p3.exists():
        return p3
    if root is not None:
        for cand in root.rglob(row["image_id"]):
            if cand.exists():
                return cand
    raise FileNotFoundError(f"Không tìm thấy ảnh: {row['image_id']}")

def compute_single_hash(row):
    try:
        p = resolve_img_path(row, target_root)
        with Image.open(p) as img:
            ph = str(imagehash.phash(img))
        if "file_hash" in row and pd.notna(row["file_hash"]) and str(row["file_hash"]).strip():
            fh = str(row["file_hash"])
        else:
            with open(p, "rb") as f:
                fh = hashlib.md5(f.read()).hexdigest()
        return ph, fh
    except Exception as e:
        print(f"Lỗi đọc ảnh {row['image_id']}: {e}")
        return "", ""

print("Đang tính toán pHash (64-bit DCT) đa luồng cho toàn bộ 21.165 ảnh...")
rows_list = [r for _, r in df_manifest.iterrows()]
with ThreadPoolExecutor(max_workers=4) as executor:
    results = list(tqdm(executor.map(compute_single_hash, rows_list), total=len(rows_list), desc="Computing pHash"))

df_manifest["phash"] = [r[0] for r in results]
df_manifest["file_hash"] = [r[1] for r in results]

# Thống kê ảnh trùng hệt theo MD5 file_hash
dup_md5 = df_manifest[df_manifest.duplicated("file_hash", keep=False)]
print(f"Số ảnh trùng hệt theo MD5 file_hash: {len(dup_md5)} ảnh ({len(dup_md5.groupby('file_hash'))} nhóm)")


In [ ]:
# 3. Thử nghiệm và phân tích 3 ngưỡng khoảng cách Hamming ứng viên (0, 4, 8)
def hex_to_bin_array(hex_str):
    val = int(hex_str, 16)
    return np.array([(val >> i) & 1 for i in range(64)], dtype=np.int8) * 2 - 1

print("Vector hoá pHash để tính toán ma trận khoảng cách Hamming...")
phash_mat = np.array([hex_to_bin_array(h) for h in df_manifest["phash"]], dtype=np.int8)

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
    def find(self, i):
        path = []
        while self.parent[i] != i:
            path.append(i)
            i = self.parent[i]
        for node in path:
            self.parent[node] = i
        return i
    def union(self, i, j):
        root_i = self.find(i)
        root_j = self.find(j)
        if root_i != root_j:
            self.parent[root_i] = root_j
            return True
        return False

candidate_thresholds = [0, 4, 8]
threshold_results = {}
chunk_size = 2000
n_samples = len(df_manifest)

working_dir = Path("/kaggle/working")
if not working_dir.exists():
    working_dir = Path(".")

for T in candidate_thresholds:
    print(f"\n--- Đang xử lý ngưỡng Hamming T = {T} ---")
    uf = UnionFind(n_samples)
    min_sim = 64 - 2 * T  # u . v >= 64 - 2T
    all_pairs = []
    
    for start_i in tqdm(range(0, n_samples, chunk_size), desc=f"Threshold {T}"):
        end_i = min(start_i + chunk_size, n_samples)
        chunk = phash_mat[start_i:end_i]
        sim_chunk = np.dot(chunk, phash_mat.T)
        
        for row_offset, i in enumerate(range(start_i, end_i)):
            sim_row = sim_chunk[row_offset]
            matching_j = np.where(sim_row >= min_sim)[0]
            for j in matching_j:
                if j > i:  # tam giác trên
                    dist = int((64 - sim_row[j]) // 2)
                    uf.union(i, j)
                    if len(all_pairs) < 500:
                        all_pairs.append((i, j, dist))
    
    # Gom nhóm theo root
    groups = defaultdict(list)
    for i in range(n_samples):
        groups[uf.find(i)].append(i)
    
    dup_groups = {k: v for k, v in groups.items() if len(v) > 1}
    n_dup_images = sum(len(v) for v in dup_groups.values())
    
    # Kiểm tra cross-class duplicates
    cross_class_groups = 0
    for grp in dup_groups.values():
        labels = set(df_manifest.iloc[grp]["label"])
        if len(labels) > 1:
            cross_class_groups += 1
            
    # Chọn tối đa 6 cặp ảnh mẫu đa dạng (ưu tiên cặp lệch nhãn nếu có)
    cross_pairs = [p for p in all_pairs if df_manifest.iloc[p[0]]["label"] != df_manifest.iloc[p[1]]["label"]]
    same_pairs = [p for p in all_pairs if df_manifest.iloc[p[0]]["label"] == df_manifest.iloc[p[1]]["label"]]
    sample_pairs = cross_pairs[:3] + same_pairs[:(6 - len(cross_pairs[:3]))]
    
    threshold_results[T] = {
        "num_duplicate_groups": len(dup_groups),
        "num_grouped_images": n_dup_images,
        "cross_class_groups": cross_class_groups,
        "sample_pairs": sample_pairs,
        "groups_map": groups
    }
    
    print(f"Kết quả T={T}: {len(dup_groups)} nhóm trùng lặp | {n_dup_images} ảnh bị gộp | {cross_class_groups} nhóm lệch nhãn")
    
    # Trực quan hóa 6 cặp ảnh mẫu
    if sample_pairs:
        n_plots = len(sample_pairs)
        fig, axes = plt.subplots(n_plots, 2, figsize=(8, 2.5 * n_plots))
        if n_plots == 1:
            axes = np.array([axes])
        for p_idx, (idx1, idx2, dist) in enumerate(sample_pairs):
            r1 = df_manifest.iloc[idx1]
            r2 = df_manifest.iloc[idx2]
            im1 = Image.open(resolve_img_path(r1, target_root))
            im2 = Image.open(resolve_img_path(r2, target_root))
            
            axes[p_idx, 0].imshow(im1, cmap="gray")
            axes[p_idx, 0].set_title(f"#{idx1}: {r1['label']} ({r1['image_id']})", fontsize=8)
            axes[p_idx, 0].axis("off")
            
            axes[p_idx, 1].imshow(im2, cmap="gray")
            axes[p_idx, 1].set_title(f"#{idx2}: {r2['label']} ({r2['image_id']})\nHamming={dist}", fontsize=8)
            axes[p_idx, 1].axis("off")
        
        plt.suptitle(f"Mẫu cặp ảnh trùng lặp (Ngưỡng Hamming T = {T})", fontsize=12, y=1.01)
        plt.tight_layout()
        plot_out = working_dir / f"duplicates_threshold_{T}.png"
        plt.savefig(plot_out, dpi=120, bbox_inches="tight")
        print(f"Đã lưu ảnh mẫu tại: {plot_out}")
        plt.show()


In [ ]:
# 4. Bảng tổng hợp đối chiếu 3 ngưỡng ứng viên
summary_rows = []
for T in candidate_thresholds:
    res = threshold_results[T]
    summary_rows.append({
        "threshold": int(T),
        "meaning": "Exact pHash" if T==0 else ("Very Similar (Hamming <= 4)" if T==4 else "Similar / Slight Shift (Hamming <= 8)"),
        "num_duplicate_groups": int(res["num_duplicate_groups"]),
        "num_grouped_images": int(res["num_grouped_images"]),
        "num_unique_images": int(len(df_manifest) - res["num_grouped_images"]),
        "cross_class_groups": int(res["cross_class_groups"]),
        "sample_pairs_count": len(res["sample_pairs"])
    })

df_comparison = pd.DataFrame(summary_rows)
print("=== BẢNG SO SÁNH 3 NGƯỠNG KHOẢNG CÁCH HAMMING ===")
print(df_comparison.to_string(index=False))

with open(working_dir / "duplicates_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_rows, f, ensure_ascii=False, indent=2)
print(f"Đã lưu duplicates_summary.json tại: {working_dir / 'duplicates_summary.json'}")


In [ ]:
# 5. Chia tập dữ liệu 70/15/15 phân tầng theo lớp và theo nhóm (StratifiedGroupKFold)
def perform_split_for_threshold(T, seed=42):
    groups_dict = threshold_results[T]["groups_map"]
    item_to_group = {}
    for grp_idx, (root, members) in enumerate(groups_dict.items()):
        gid = f"G{grp_idx:05d}"
        for m in members:
            item_to_group[m] = gid
    
    df = df_manifest.copy()
    df["group_id"] = [item_to_group[i] for i in range(len(df))]
    
    # Chia 20 fold: 14 fold (70%) train, 3 fold (15%) val, 3 fold (15%) test
    sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=seed)
    folds = np.zeros(len(df), dtype=int)
    for fold_id, (_, test_idx) in enumerate(sgkf.split(df, df["label"], df["group_id"])):
        folds[test_idx] = fold_id
        
    splits = []
    for f in folds:
        if f < 14:
            splits.append("train")
        elif f < 17:
            splits.append("val")
        else:
            splits.append("test")
    df["split"] = splits
    
    cols = ["image_id", "label", "rel_image_path", "rel_mask_path", "has_mask", "group_id", "split", "file_hash", "phash"]
    return df[cols]

candidate_dir = working_dir / "candidate_splits"
candidate_dir.mkdir(parents=True, exist_ok=True)

for T in candidate_thresholds:
    df_split = perform_split_for_threshold(T, seed=42)
    out_path = candidate_dir / f"split_th{T}.csv"
    df_split.to_csv(out_path, index=False)
    print(f"Đã xuất candidate split T={T} tại: {out_path}")
    
    split_dist = pd.crosstab(df_split["label"], df_split["split"], normalize="columns") * 100
    print(f"\nPhần trăm phân bố các lớp theo split (T={T}):")
    print(split_dist.round(2))

# Lưu mặc định candidate T=0 ra working để nghiệm thu sơ bộ
df_default = perform_split_for_threshold(0, seed=42)
default_split_path = working_dir / "split_v1.csv"
df_default.to_csv(default_split_path, index=False)
sha256_hash = hashlib.sha256(open(default_split_path, "rb").read()).hexdigest()
with open(working_dir / "split_v1.sha256", "w", encoding="utf-8") as f:
    f.write(f"{sha256_hash}  split_v1.csv\n")
print(f"\nĐã xuất bản mẫu split_v1.csv (T=0) và split_v1.sha256: {sha256_hash}")


## 6. Ghi chú về Metadata Bệnh nhân & Nguyên tắc Khoá tập Test

1. **Giới hạn về định danh bệnh nhân (Patient ID Limitation):**
   - Bộ dữ liệu `COVID-19 Radiography Database` chỉ cung cấp tên file theo số thứ tự (ví dụ: `COVID-1.png`, `Normal-1.png`), **không cung cấp metadata mã bệnh nhân** (`patient_id`).
   - Việc lọc trùng hệt (MD5) và gần trùng (pHash) là giải pháp kỹ thuật tối ưu nhất có thể áp dụng để gom các ảnh giống nhau vào cùng một split.
   - **Giới hạn khoa học thừa nhận:** Nguy cơ rò rỉ giữa các ảnh chụp khác nhau của cùng một bệnh nhân (ví dụ: chụp cách nhau vài ngày với góc chụp hoặc tư thế khác nhau) không thể loại trừ hoàn toàn do thiếu metadata gốc.

2. **Quy tắc khoá nghiêm ngặt tập Test (Test Set Locking):**
   - File `splits/split_v1.csv` đã được chia cố định và khoá bằng mã băm `split_v1.sha256`.
   - Tập `test` (15% ~ 3.175 ảnh) **TUYỆT ĐỐI KHÔNG ĐƯỢC PHÉP TRUY CẬP** trong các Tuần 4 đến 9 (huấn luyện Baseline, RQ1, RQ2, tinh chỉnh siêu tham số hay lựa chọn mô hình).
   - Tập `test` chỉ được mở ra đánh giá **DUY NHẤT 1 LẦN tại Tuần 10** (`10_final_test_eval.ipynb`) trên mô hình tối ưu cuối cùng.
